In [1]:
import sys, os
sys.path.insert(0, '../../utils')


In [2]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, PercentFormatter, MaxNLocator
from plot_utils import ex_color
from ensembles import require_ensemble_results, shared_input_by_size


In [3]:
# ── how this figure is drawn ─────────────────────────────────────────────────
# The panel drawing lives here rather than in utils/, so the notebook that produces
# figure S14 is readable end to end. utils/ensembles.py holds the analysis: detection,
# the matched controls and the bootstraps, all run once by scripts/ensemble_run.py.

import matplotlib
from matplotlib.colors import to_rgb


CONTROL_COLOR  = "#999999"       # histogram fill (rendered with alpha=0.7)


def darken(color, factor=0.65):
    """A darker shade of `color` — for lines that must read against their own fill."""
    r, g, b = to_rgb(color)
    return (r * factor, g * factor, b * factor)


def star_below_legend(ax, star, fontsize=None, pad=0.02, color='black'):
    """Draw a significance star on its own line just under the legend box.

    Keeps the star out of the legend labels (where it crowds the value text)
    while still anchoring it to the legend rather than to a hand-tuned axes
    position: the star is left-aligned with the legend's label column and sits
    `pad` (axes fraction) below the bottom-most label, so it reads as one more
    line of the legend block regardless of how many entries there are.

    The anchor is the last label's text box, not `legend.get_window_extent()` —
    the latter includes the legend's border padding and handle column, which
    puts the star noticeably low and to the left of the labels.

    Must be called after `ax.legend(...)`. A legend computes its layout only
    while being drawn, so its artists have no meaningful extent before the first
    draw — hence the explicit `canvas.draw()` rather than reusing a renderer.
    """
    leg = ax.get_legend()
    if not star or leg is None:
        return None
    canvas = ax.get_figure().canvas
    canvas.draw()
    last = leg.get_texts()[-1]
    bb = last.get_window_extent(canvas.get_renderer()).transformed(ax.transAxes.inverted())
    if fontsize is None:
        fontsize = last.get_fontsize()
    return ax.text(bb.x0, bb.y0 - pad, star, transform=ax.transAxes,
                   ha='left', va='top', fontsize=fontsize, color=color)


def plot_null_hist_panel(ax, null_vals, obs_val, global_val, ctrl_color, obs_color,
                         obs_label, global_label, ctrl_label='Control',
                         bins=20, xlabel='', ylabel='Count', star=None,
                         legend_loc='upper left', legend_bbox=(0, 1.05),
                         headroom=1.5, marker_pad=1.03, ctrl_mean_color=None,
                         ctrl_mean_fmt='.3g', trim_yticks=True):
    """Null-distribution histogram + observed / control-mean vertical markers.

    Shared by the metric-B (connection probability) and metric-A (% synapses on
    spines) panels, which differ only in data, bins and x label.

    `ax.axvline` spans the full axes height, so with a `best`-placed legend the
    marker lines and the tallest bars run straight through the legend text. Here
    the markers are drawn with `vlines` capped just above the tallest bar
    (`marker_pad`) and the y limit is opened to `headroom` × that height, so the
    legend sits in a band that no artist reaches into.

    `star` (e.g. from `stats_corr.p_to_stars`) is drawn between the ensemble
    and control-mean vlines, including when it is 'ns'.

    The control-mean dashed line sits on top of the control bars, so drawing it
    in `ctrl_color` makes it near-invisible however opaque it is — same hue, and
    the bars behind it are only alpha-lightened. It is drawn in a darkened shade
    (`ctrl_mean_color`, default `darken(ctrl_color)`) so it reads as the same
    grey but stands off its own histogram.

    `trim_yticks` drops the y ticks that fall inside the headroom band. The band
    is deliberate — it is what keeps the legend off the bars — but no bar ever
    reaches it, so ticks up there label empty space. The limit is unchanged.

    Returns the histogram counts.
    """
    ctrl_mean = np.mean(null_vals)
    ctrl_leg_label = f'{ctrl_label} (mean = {ctrl_mean:{ctrl_mean_fmt}})'
    counts, _, _ = ax.hist(null_vals, bins=bins, color=ctrl_color, alpha=0.7,
                           label=ctrl_leg_label)
    y_top = counts.max() * marker_pad
    ax.vlines(obs_val, 0, y_top, color=obs_color, lw=2, label=obs_label)
    ax.vlines(global_val, 0, y_top, color='k', lw=1.5, ls='--', label=global_label)
    ax.vlines(ctrl_mean, 0, y_top, lw=2, ls='--',
              color=darken(ctrl_color) if ctrl_mean_color is None else ctrl_mean_color)
    ax.set_ylim(0, counts.max() * headroom)
    if trim_yticks:
        # set_yticks can rescale the view, so restore the limit afterwards —
        # the point is to lose the ticks, not the headroom they sat in.
        _ylim = ax.get_ylim()
        ax.set_yticks([t for t in ax.get_yticks() if 0 <= t <= counts.max()])
        ax.set_ylim(_ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # Ensembles leads the legend. Draw order would put Control first — the
    # histogram has to be drawn before the vlines that cap to its height — but
    # ensembles is the result the panel is about, so the entries are reordered
    # rather than the drawing, which would change the z-order too.
    _h, _l = ax.get_legend_handles_labels()
    _by_label = dict(zip(_l, _h))
    _order = [lb for lb in (obs_label, ctrl_leg_label, global_label)
              if lb in _by_label]
    ax.legend([_by_label[lb] for lb in _order], _order,
              frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    if star:
        x_lo = min(obs_val, ctrl_mean)
        x_hi = max(obs_val, ctrl_mean)
        x_mid = (x_lo + x_hi) / 2
        y_star = y_top
        ax.annotate('', xy=(x_hi, y_star), xytext=(x_lo, y_star),
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.75, alpha=0.8),
                    annotation_clip=False)
        ax.text(x_mid, y_star, star, ha='center', va='bottom',
                fontsize=matplotlib.rcParams.get('font.size', 12), color='black',
                clip_on=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    return counts


def plot_shared_input_panel(ax, real_df, ctrl_df, ctrl_color, ens_color,
                            xlim=None, pct=95, floor=5,
                            inset_xlim=None, inset_pct=95,
                            inset_bounds=(0.55, 0.38, 0.43, 0.52),
                            inset_fontsize=8, star=None, inset_star=None,
                            legend_loc='upper left',
                            legend_bbox=(0.10, 1.04), verbose=True):
    """Shared-input histogram (I-only) with an inset for the E-only counts.

    Both distributions are drawn as unfilled `stairs` outlines with integer bins
    and `density=True`, so ensembles (n≈40) and control (n≈40k) are comparable.

    The main axes show `shared_inh`, the intersection of the members' inhibitory
    pre-synaptic sets — the flavour that carries the effect. The inset shows
    `shared_ex`, which lives on a much smaller range and would otherwise collapse
    into the first two bins of the main axes.

    Both x ranges are set the same way: the `pct` / `inset_pct` percentile of
    the *pooled* real+control values, rounded up and floored at `floor` so the
    axis never degenerates (the shared_ex p95 is 1 for this dataset). Because
    control dominates the pool (n≈40k vs 41), the cut follows the control
    distribution and clips the longer ensemble tail — the fraction of each
    group actually shown is logged. Pass `xlim` / `inset_xlim` to override.

    `star` / `inset_star` (e.g. from `stats_corr.p_to_stars`) annotate the two
    ensembles-vs-control tests, and are shown even when 'ns'. The main one is
    drawn under the legend by `star_below_legend`; the inset has no legend of
    its own, so its star goes in the inset's empty top-right corner.

    Returns (ax_inset, info_dict) where info_dict carries the max/percentile
    values that were logged.
    """
    ens_inh  = real_df['shared_inh'].values.astype(float)
    ctrl_inh = ctrl_df['shared_inh'].values.astype(float)
    ens_ex  = real_df['shared_ex'].values.astype(float)
    ctrl_ex = ctrl_df['shared_ex'].values.astype(float)

    def _stairs(axis, vals, color, label, xmax, lw):
        bins = np.arange(-0.5, xmax + 1.5, 1.0)
        counts, edges = np.histogram(vals, bins=bins, density=True)
        axis.stairs(counts, edges, color=color, linewidth=lw, label=label)

    def _pct_xlim(pooled, q, explicit):
        """Percentile-based integer x limit; `explicit` wins when given."""
        p = float(np.percentile(pooled, q))
        return (explicit if explicit is not None else int(max(floor, np.ceil(p)))), p

    # ── main axes: I-only shared input ──
    pooled_inh   = np.concatenate([ens_inh, ctrl_inh])
    xlim, p_inh  = _pct_xlim(pooled_inh, pct, xlim)
    for vals, color, label in [(ctrl_inh, ctrl_color, 'Control'),
                               (ens_inh,  ens_color,  'Ensembles')]:
        _stairs(ax, vals, color, label, xlim, 1.5)
    ax.set_xlim(-0.5, xlim + 0.5)
    ax.set_xticks(np.arange(0, xlim + 1, 5 if xlim > 15 else 2))
    ax.set_xlabel('Shared Inh input')
    ax.set_ylabel('Probability')
    # Legend nudged right of the x=0 spike so it clears both the tall first bin
    # and the inset, which occupies the upper-right quadrant.
    # Control is *drawn* first so the ensemble outline sits on top of it, but the
    # legend leads with Ensembles, matching the other panels' obs-then-null order.
    _h, _l = ax.get_legend_handles_labels()
    _order = [_l.index('Ensembles'), _l.index('Control')]
    ax.legend([_h[i] for i in _order], [_l[i] for i in _order],
              frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    star_below_legend(ax, star)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))

    # ── inset: E-only shared input ──
    pooled_ex            = np.concatenate([ens_ex, ctrl_ex])
    inset_xlim, p_ex     = _pct_xlim(pooled_ex, inset_pct, inset_xlim)

    ax_in = ax.inset_axes(inset_bounds)
    for vals, color in [(ctrl_ex, ctrl_color), (ens_ex, ens_color)]:
        _stairs(ax_in, vals, color, None, inset_xlim, 1.2)
    ax_in.set_xlim(-0.5, inset_xlim + 0.5)
    ax_in.set_xticks(np.arange(0, inset_xlim + 1))
    ax_in.set_xlabel('Shared Ex input', fontsize=inset_fontsize, labelpad=1)
    ax_in.set_ylabel('Probability', fontsize=inset_fontsize, labelpad=1)
    if inset_star:
        # The inset shares the main legend, so there is no legend box of its own
        # to sit under; its star goes in the inset's own top-right corner, which
        # the decaying distribution always leaves empty.
        ax_in.text(0.97, 0.95, inset_star, transform=ax_in.transAxes,
                   ha='right', va='top', fontsize=inset_fontsize + 2)
    ax_in.tick_params(labelsize=inset_fontsize - 1, length=2, pad=1)
    ax_in.spines[['top', 'right']].set_visible(False)
    ax_in.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    ax_in.patch.set_alpha(0)

    info = {'max_shared_ex_ens':  float(ens_ex.max()),
            'max_shared_ex_ctrl': float(ctrl_ex.max()),
            'max_shared_inh_ens':  float(ens_inh.max()),
            'max_shared_inh_ctrl': float(ctrl_inh.max()),
            f'p{pct}_shared_inh':      p_inh,
            f'p{inset_pct}_shared_ex': p_ex,
            'xlim': xlim, 'inset_xlim': inset_xlim}
    if verbose:
        # plain ASCII in the log lines: this also runs from consoles with a
        # non-UTF-8 codepage, where a literal arrow raises UnicodeEncodeError.
        for name, ens_v, ctrl_v, q, p, lim in [
            ('shared_inh', ens_inh, ctrl_inh, pct,       p_inh, xlim),
            ('shared_ex', ens_ex, ctrl_ex, inset_pct, p_ex, inset_xlim),
        ]:
            print(f"{name} max: ensembles={ens_v.max():.0f}  control={ctrl_v.max():.0f}  "
                  f"(pooled p{q}={p:.0f} -> x range 0-{lim}; shown: "
                  f"{(ens_v <= lim).mean():.1%} of ensembles, "
                  f"{(ctrl_v <= lim).mean():.1%} of control)")
    return ax_in, info


def plot_shared_input_by_size_panel(ax, d_size, ctrl_color, ens_color,
                                    metric='shared_inh', show_fold=True,
                                    fold_only_reliable=False,
                                    fold_fmt='{:.2f}×', fold_fontsize=8,
                                    show_inh_pct=False, inh_metrics=('shared_inh', 'shared_ex'),
                                    inh_fmt='{:.0f}% Inh',
                                    legend_loc='upper right', legend_bbox=None,
                                    bar_width=0.4,
                                    xlabel='Ensemble size (neurons)',
                                    ylabel='Shared I inputs per ensemble'):
    """Per-ensemble-size shared-input bars, in the figure-6 colour convention.

    Draws one measure onto an axes the caller already owns, in the red/grey
    ensemble/control convention, so it can sit inside a figure row.

    `d_size` is the output of `shared_input_by_size`; cumulative rows ('n>=3', 'n>=4')
    are dropped since they have no position on a size axis. Bar height is the
    **per-ensemble mean** (`value` / `value_null`), not the pooled sum, so bars
    stay comparable across sizes holding different numbers of ensembles.

    Note for the caption: shared input is an *intersection* over members, so it
    falls with ensemble size by construction — an interneuron must contact every
    member to count. The readable quantity is the ensemble/control gap, which is
    why the fold is annotated above each pair rather than left to the reader.

    That construction is also why `fold_only_reliable` exists. Once the ensembles
    are large enough, no interneuron reaches every member: both bars go to zero
    and the fold becomes 0/0.002 = '0.00x' printed above an invisible bar, which
    reads as a broken panel rather than as the real finding that the intersection
    is empty. `shared_input_by_size` already marks those rows `reliable=False` (null
    mean under its `min_null`); set this True to keep their bars but drop their
    annotations. Default False so existing callers are unchanged.

    `show_inh_pct` adds a second annotation line under the fold: what fraction of
    the shared input at that size is inhibitory, inh / (inh + ex) over the *observed*
    counts (`inh_metrics`, in that order). A presynaptic cell has exactly one
    clf_type, so the two counts partition the shared inputs and no null enters —
    this is the E/I composition table of ensembles.md §6.5, drawn in place. Sizes
    where the frame has no shared input at all (0/0) get the fold line only.

    Returns the per-size sub-frame actually plotted.
    """
    d = d_size[(d_size['metric'] == metric)
               & ~d_size['size'].astype(str).str.startswith('n>=')].copy()
    if d.empty:
        raise ValueError(f'no per-size rows for metric {metric!r} in d_size')
    d['size_i'] = d['size'].astype(int)
    d = d.sort_values('size_i')

    x = np.arange(len(d))
    ax.bar(x - bar_width / 2, d['value'], width=bar_width,
           color=ens_color, alpha=0.7, label='Ensembles')
    ax.bar(x + bar_width / 2, d['value_null'], width=bar_width,
           color=ctrl_color, alpha=0.8, label='Control')

    if show_fold:
        # Keyed on the raw `size` label so it lines up with `d` before the int cast.
        pct_by_size = {}
        if show_inh_pct:
            counts = {m: (d_size[d_size['metric'] == m]
                          .set_index(d_size.loc[d_size['metric'] == m, 'size']
                                     .astype(str))['obs_num'].to_dict())
                      for m in inh_metrics}
            for sz in d['size'].astype(str):
                inh, ex = (counts[m].get(sz, np.nan) for m in inh_metrics)
                if np.isfinite(inh) and np.isfinite(ex) and (inh + ex) > 0:
                    pct_by_size[sz] = 100.0 * inh / (inh + ex)

        # Annotated above the taller of the pair so the label never sits on a bar.
        # `reliable` may be absent from hand-built frames, hence the getattr.
        for xi, row in zip(x, d.itertuples()):
            if fold_only_reliable and not getattr(row, 'reliable', True):
                continue
            if np.isfinite(row.fold):
                txt = fold_fmt.format(row.fold)
                pct = pct_by_size.get(str(row.size))
                if pct is not None:
                    txt += '\n' + inh_fmt.format(pct)
                ax.annotate(txt, (xi, max(row.value, row.value_null)),
                            textcoords='offset points', xytext=(0, 2),
                            ha='center', va='bottom', fontsize=fold_fontsize)
        # Two annotation lines need roughly twice the headroom over the tallest bar.
        pad = 1.30 if pct_by_size else 1.18
        ax.set_ylim(0, max(d[['value', 'value_null']].to_numpy().max() * pad, 1e-9))

    ax.set_xticks(x)
    ax.set_xticklabels(d['size_i'])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # legend_bbox nudges the legend off the tallest bar pair: at a narrow axes
    # width 'upper right' alone puts the swatches right against them.
    ax.legend(frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    ax.spines[['top', 'right']].set_visible(False)
    return d

In [4]:
# The ecker run pickle. Not downloaded — produce it once with
# `python scripts/ensemble_run.py` (which first needs the two activity H5 files).
with open(require_ensemble_results('ecker'), 'rb') as f:
    results_sup = pickle.load(f)

all_ensembles_df_sup = results_sup['all_ensembles_df']
metric_a_sup         = results_sup['spine_targeting']
metric_b_sup         = results_sup['connection_probability']
metric_c_sup         = results_sup['shared_input']
print(f'Ecker: {len(all_ensembles_df_sup)} ensembles across '
      f'{all_ensembles_df_sup[["session", "scan_idx"]].drop_duplicates().shape[0]} scans, '
      f'sizes {all_ensembles_df_sup["n_members"].min()}-{all_ensembles_df_sup["n_members"].max()}')

# Same 1000-replicate bootstrap as the main figure, so cached the same way —
# `del d_size_sup` to force a recompute.
try:
    d_size_sup
except NameError:
    d_size_sup = shared_input_by_size(results_sup)

# Column-wide baselines, independent of which detector ran.
# Panel E: global I->E spine fraction, 16,668 / 72,207 tagged I->E synapses.
P_GLOBAL_IE_SPINE = 16668 / 72207
# Panel C: 29,735 / 36,176 over ALL E->E synapses, not the tagged-only denominator.
_p_global_c3 = 29735 / 36176
# Panel B: over all ordered pairs of the 1,188 column excitatory neurons.
global_EE_conn_prob = 36176 / (1188 * 1187)
print(f'Global I->E spine fraction:         {P_GLOBAL_IE_SPINE:.4f}')
print(f'Global E->E spine fraction:         {_p_global_c3:.4f}')
print(f'Global E->E connection probability: {global_EE_conn_prob:.6f}')

_sp_ii_sup = metric_c_sup['shared_inh_spine_frac']
print(f'B  conn prob      obs={metric_b_sup["p_obs"]:.4f}  '
      f'null={np.mean(metric_b_sup["valid_null"]):.4f}  {metric_b_sup["star"]}')
print(f'C  spine frac     obs={metric_a_sup["p_obs"]:.4f}  '
      f'null={np.mean(metric_a_sup["valid_null"]):.4f}  {metric_a_sup["star"]}')
print(f'E  shared I spine obs={_sp_ii_sup["obs"]:.4f}  null={np.mean(_sp_ii_sup["null"]):.4f}  '
      f'p={_sp_ii_sup["p_emp"]:.4f} {_sp_ii_sup["star"]}  '
      f'n={_sp_ii_sup["n_num"]}/{_sp_ii_sup["n_den"]}')


Ecker: 181 ensembles across 14 scans, sizes 2-15
Global I->E spine fraction:         0.2308
Global E->E spine fraction:         0.8220
Global E->E connection probability: 0.025654
B  conn prob      obs=0.0618  null=0.0294  ***
C  spine frac     obs=0.9182  null=0.7375  ***
E  shared I spine obs=0.2511  null=0.2245  p=0.0004 ***  n=2507/9986


In [5]:
# control font size
plt.rcParams['font.size'] = 19
plt.rcParams['legend.fontsize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['font.family'] = 'Arial'

LABEL_FONTSIZE   = 14   # schematic captions ('Ensemble' / 'Control' / 'Shared input' / …)
LEG_FONTSIZE     = 14   # inner legends in B / C / D
letter_font_size = 24   # 'A' 'B' 'C' 'D' 'E' panel letters
INSET_FONTSIZE   = 13   # Panel D — shared-E-input inset labels/ticks/star
FOLD_FONTSIZE    = 11.5   # Panel D — 'N.NNx' fold annotations above by-size bars
RASTER_FONTSIZE  = 14   # Panel A — every raster text (row labels, clip nums, scale bar, …)


In [6]:
# =============================================================================
# LAYOUT CONFIG — Fig 6 Sup. Data panels only, so every knob here is a width
# ratio or a margin; no schematic geometry to tune.
# =============================================================================
SUP_FIG_SIZE   = (18, 9.0)
SUP_ROW_HEIGHTS = [1.0, 1.0]
SUP_ROW_GAP     = 0.07

# Row 1 — A | B | C. Equal thirds: all three are single histograms.
SUP_R1_SPLIT    = [1.0, 1.0, 1.0]
SUP_R1_GAP      = 0.05

# Row 2 — D (shared-I hist) | by-size bars | E. With the schematics gone the
# by-size bars finally get a real share of the width instead of the main
# figure's 3.1 : 1 squeeze — 13 bar pairs plus fold labels need the room.
SUP_R2_SPLIT    = [1.15, 1.0, 1.15]
SUP_R2_GAP      = 0.05

SUP_HIST_MARG   = dict(left=0.15, right=0.96, top=0.86, bottom=0.16)
SUP_SIZE_MARG   = dict(left=0.15, right=0.97, top=0.86, bottom=0.16)
SUP_LETTER_XY   = (0.015, 0.99)   # panel-letter position inside each subfigure
SUP_LETTER_FS   = 20              # panel letters. Below the main figure's 24: this
                                  #   figure is 18 x 9 against 20.5 x 14, so the same
                                  #   point size reads bigger once both are scaled to a
                                  #   page. Bold, as in the main figure.
SUP_FOLD_FS     = 13              # by-size fold annotations; larger than the
                                  #   main figure's 11.5 now that the bars are wide
SUP_SIZE_MAX    = 6             # keep 6
                                 
# =============================================================================

fig_sup = plt.figure(figsize=SUP_FIG_SIZE, dpi=600)
sfs_sup = fig_sup.subfigures(2, 1, height_ratios=SUP_ROW_HEIGHTS, hspace=SUP_ROW_GAP)

# =========================================================================
# ROW 1 — A: ensemble sizes | B: connection probability | C: % on spines
# =========================================================================
sf_sup_a, sf_sup_b, sf_sup_c = sfs_sup[0].subfigures(1, 3, width_ratios=SUP_R1_SPLIT,
                                                     wspace=SUP_R1_GAP)

# ---- A ----
ax_sup_a = sf_sup_a.subplots(1, 1)
sf_sup_a.subplots_adjust(**SUP_HIST_MARG)
sns.histplot(data=all_ensembles_df_sup, x='n_members', bins=30,
             color=ex_color, alpha=0.7, discrete=True, ax=ax_sup_a)
ax_sup_a.set_xlabel('Ensemble size (neurons)')
ax_sup_a.set_ylabel('Count')
ax_sup_a.spines[['top', 'right']].set_visible(False)
# One tick per observed size while that stays legible; the detector can return a
# long tail of sizes, in which case the auto locator is the better choice.
_xt_sup = np.sort(all_ensembles_df_sup['n_members'].unique())
if len(_xt_sup) < 15:
    ax_sup_a.set_xticks(_xt_sup)
    ax_sup_a.set_xticklabels(_xt_sup)
ax_sup_a.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
ax_sup_a.yaxis.set_major_formatter(FormatStrFormatter('%g'))
sf_sup_a.text(*SUP_LETTER_XY, 'A', fontsize=SUP_LETTER_FS, fontweight='bold', va='top')

# ---- B ----
ax_sup_b = sf_sup_b.subplots(1, 1)
sf_sup_b.subplots_adjust(**SUP_HIST_MARG)
plot_null_hist_panel(
    ax_sup_b, metric_b_sup['valid_null'],
    obs_val=metric_b_sup['p_obs'], global_val=global_EE_conn_prob,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({metric_b_sup["p_obs"]:.1%})',
    global_label=f'Baseline ({global_EE_conn_prob:.1%})',
    ctrl_mean_fmt='.1%', star=metric_b_sup['star'], headroom=1.5,
    legend_bbox=(0, 1.125), bins=20, xlabel='Connection probability (%)')
ax_sup_b.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
sf_sup_b.text(*SUP_LETTER_XY, 'B', fontsize=SUP_LETTER_FS, fontweight='bold', va='top')

# ---- C ----
ax_sup_c = sf_sup_c.subplots(1, 1)
sf_sup_c.subplots_adjust(**SUP_HIST_MARG)
plot_null_hist_panel(
    ax_sup_c, metric_a_sup['valid_null'],
    obs_val=metric_a_sup['p_obs'], global_val=_p_global_c3,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({metric_a_sup["p_obs"]:.0%})',
    global_label=f'Baseline ({_p_global_c3:.0%})',
    ctrl_mean_fmt='.0%', star=metric_a_sup['star'], headroom=1.5,
    legend_bbox=(0, 1.125), bins=30, xlabel='% of synapses on spines')
ax_sup_c.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
# Same anchoring as the main figure: 100% is the natural right edge of a
# percentage axis, and the auto locator stops at 80.
ax_sup_c.set_xticks([0.6, 0.8, 1.0])
ax_sup_c.set_xlim(right=max(ax_sup_c.get_xlim()[1], 1.0))
sf_sup_c.text(*SUP_LETTER_XY, 'C', fontsize=SUP_LETTER_FS, fontweight='bold', va='top')

# =========================================================================
# ROW 2 — D: shared I input (+ by-size) | E: spine fraction of shared I input
# =========================================================================
sf_sup_d, sf_sup_dsz, sf_sup_e = sfs_sup[1].subfigures(1, 3, width_ratios=SUP_R2_SPLIT,
                                                       wspace=SUP_R2_GAP)

# ---- D ----
ax_sup_d = sf_sup_d.subplots(1, 1)
sf_sup_d.subplots_adjust(**SUP_HIST_MARG)
ax_sup_d_ee, _shared_info_sup = plot_shared_input_panel(
    ax_sup_d, metric_c_sup['real_df'], metric_c_sup['ctrl_df'],
    ctrl_color=CONTROL_COLOR, ens_color=ex_color, inset_pct=99, pct=99,
    inset_fontsize=INSET_FONTSIZE, legend_bbox=(0.03, 1.04),
    star=None, inset_star=None, verbose=False)
sf_sup_d.text(*SUP_LETTER_XY, 'D', fontsize=SUP_LETTER_FS, fontweight='bold', va='top')

# ---- D, by size (shares the D letter, as in the main figure) ----
# Shared input is an intersection over members, so it falls with ensemble size
# by construction — an interneuron must contact every member to count. The fold
# annotations carry the content.
ax_sup_dsz = sf_sup_dsz.subplots(1, 1)
sf_sup_dsz.subplots_adjust(**SUP_SIZE_MARG)
# The cumulative 'n>=3' / 'n>=4' rows are dropped by the panel itself; the size
# cap is applied here so it stays a layout choice, not a change to d_size_sup.
_d_size_sup_cut = d_size_sup[d_size_sup['size'].astype(str).str.startswith('n>=')
                             | (pd.to_numeric(d_size_sup['size'], errors='coerce')
                                <= SUP_SIZE_MAX)]
plot_shared_input_by_size_panel(
    ax_sup_dsz, _d_size_sup_cut, ctrl_color=CONTROL_COLOR, ens_color=ex_color,
    metric='shared_inh', ylabel='Mean shared Inh inputs',
    fold_only_reliable=True, show_inh_pct=True,
    fold_fontsize=SUP_FOLD_FS, legend_bbox=(1.02, 1.02))
ax_sup_dsz.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax_sup_dsz.yaxis.set_major_formatter(FormatStrFormatter('%g'))

# ---- E ----
ax_sup_e = sf_sup_e.subplots(1, 1)
sf_sup_e.subplots_adjust(**SUP_HIST_MARG)
plot_null_hist_panel(
    ax_sup_e, _sp_ii_sup['null'],
    obs_val=_sp_ii_sup['obs'], global_val=P_GLOBAL_IE_SPINE,
    ctrl_color=CONTROL_COLOR, obs_color=ex_color,
    obs_label=f'Ensembles ({_sp_ii_sup["obs"]:.1%})',
    global_label=f'Baseline ({P_GLOBAL_IE_SPINE:.1%})',
    ctrl_mean_fmt='.0%', star=_sp_ii_sup['star'], headroom=1.5,
    legend_bbox=(0, 1.125), bins=30,
    xlabel='% of shared Inh synapses on spines')
ax_sup_e.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
sf_sup_e.text(*SUP_LETTER_XY, 'E', fontsize=SUP_LETTER_FS, fontweight='bold', va='top')
plt.savefig('fig_s14.pdf', format='pdf', bbox_inches='tight')
